# P31 training on Kaggle

1. Settings → Accelerator: **GPU T4 x2** or **P100**, Internet: **on**.
2. Add data → search **msambare/fer2013** (appears at `/kaggle/input/fer2013`).
3. Optional: upload RAF-DB as a private Kaggle dataset named `rafdb` for the cross-dataset test.
4. Set `CONFIG` below to any file name in `configs/` (e.g. `cnn`, `cnn_long`, `resnet`, `vit`).
   Results go to `results/<CONFIG>/`, so every experiment gets its own folder.
5. Save Version → **Save & Run All (Commit)**, then download `results_<CONFIG>.zip` from the Output tab.

In [ ]:
REPO_URL = "https://github.com/Neeraj-NP/P31-facial-expression-recognition.git"
CONFIG = "cnn"   # any configs/<name>.yaml: cnn, cnn_long, cnn_lightreg, cnn_noweights, cnn_noaug, resnet, vit, ...
BRANCH = "main"  # set to a branch name to test configs that are not merged yet

# Kaggle mounts inputs at /kaggle/input/<name> or /kaggle/input/datasets/<owner>/<name>,
# so locate each dataset by a folder it must contain instead of hard-coding the path.
from pathlib import Path
def find_root(marker, levels_up, base="/kaggle/input"):
    hits = sorted(Path(base).rglob(marker))
    return str(hits[0].parents[levels_up - 1]) if hits else None

FER_ROOT = find_root("train/happy", 2)    # <root>/train/happy
RAF_ROOT = find_root("EmoLabel", 1) or "/kaggle/input/rafdb"   # <root>/EmoLabel; ignored if not attached
assert FER_ROOT, "FER-2013 not found: Add Input -> https://www.kaggle.com/datasets/msambare/fer2013"
print("FER_ROOT =", FER_ROOT, "| RAF_ROOT =", RAF_ROOT)

In [ ]:
!git clone -q -b $BRANCH $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q timm grad-cam
import torch
print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Settings -> Accelerator -> GPU T4 x2 (needs phone verification), then restart.")

In [ ]:
# point the config at Kaggle's read-only input folders
import yaml
cfg = yaml.safe_load(open(f"configs/{CONFIG}.yaml"))
cfg["data"].update(fer2013_root=FER_ROOT, rafdb_root=RAF_ROOT, num_workers=2)
yaml.safe_dump(cfg, open(f"configs/{CONFIG}_kaggle.yaml", "w"))
cfg

In [ ]:
!python -m src.train --config configs/{CONFIG}_kaggle.yaml --run-name {CONFIG}

In [ ]:
!python -m src.evaluate --run results/{CONFIG} --dataset fer2013
!python -m src.explain --run results/{CONFIG} --dataset fer2013

In [ ]:
# cross-dataset test + subgroup audit (needs RAF-DB attached)
import os
if os.path.exists(RAF_ROOT):
    !python -m src.evaluate --run results/{CONFIG} --dataset rafdb
    !python -m src.explain --run results/{CONFIG} --dataset rafdb --group-by gender
else:
    print("RAF-DB not attached, skipping")

In [ ]:
!cd /kaggle/working && zip -qr results_{CONFIG}.zip repo/results/{CONFIG}
print("download results_" + CONFIG + ".zip from the Output tab")